In [2]:
# ============================================================
# CELL 1: Mount Drive & Cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets transformers torchvision scikit-learn tqdm Pillow

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 2: Import & Config
# ============================================================
import os, json, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLNetTokenizer, XLNetModel
from torchvision import models, transforms
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR   = "/content/drive/MyDrive/SpotFakePlus_IFND"
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-5
MAX_LEN    = 256

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device  : {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

Device  : cuda
Save dir: /content/drive/MyDrive/SpotFakePlus_IFND


In [4]:
# ============================================================
# CELL 3: Load & Khám phá dataset
# ============================================================
print("Đang tải dataset Nhat243/IFND-multimodal ...")
raw = load_dataset("Nhat243/IFND-multimodal")
print(raw)
print("\nFeatures:", raw["train"].features)
print("\nMẫu đầu:")
sample = raw["train"][0]
for k, v in sample.items():
    print(f"  {k}: {str(v)[:100]}")

# Xác định test splits
test_splits = [k for k in raw.keys() if k not in ("train", "validation")]
if not test_splits:
    # Nếu không có test split riêng thì dùng validation làm test
    test_splits = ["validation"]
print(f"\nTest splits: {test_splits}")

# Thống kê label
for split in ["train", "validation"]:
    if split in raw:
        labels = raw[split]["label"] if "label" in raw[split].features \
                 else raw[split]["labels"]
        unique, counts = np.unique(labels, return_counts=True)
        print(f"\n[{split}] Label distribution:")
        for u, c in zip(unique, counts):
            print(f"  Label {u}: {c:,} mẫu")

Đang tải dataset Nhat243/IFND-multimodal ...


README.md:   0%|          | 0.00/613 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/435M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/432M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/112M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})

Features: {'id': Value('int64'), 'text': Value('string'), 'image': Image(mode=None, decode=True), 'label': Value('int64')}

Mẫu đầu:
  id: 3851
  text: HP health minister urges people who have recovered from Covid to visit isolation wards & boost moral
  image: <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=450x250 at 0x7C8DD372B980>
  label: 1

Test splits: ['test']

[train] Label distribution:
  Label 0: 1,969 mẫu
  Label 1: 6,447 mẫu

[validation] Label distribution:
  Label 0: 246 mẫu
  Label 1: 806 mẫu


In [9]:
# ============================================================
# CELL 4 (FIX): Dataset class — xử lý ảnh corrupt
# ============================================================
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

IMG_TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def get_text(item):
    return (item.get("text")    or item.get("caption") or
            item.get("title")   or item.get("content") or
            item.get("article") or item.get("statement") or "")

def get_label(item):
    label = (item.get("label") if item.get("label") is not None
             else item.get("labels", 0))
    return int(label)

class IFNDDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=MAX_LEN, is_train=False):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.transform = IMG_TRANSFORM_TRAIN if is_train else IMG_TRANSFORM

        # Tắt decode ảnh tự động của HuggingFace
        # để tự xử lý lỗi corrupt bên dưới
        self.data = self.data.cast_column(
            "image",
            datasets.features.Image(decode=False)
        )

    def __len__(self):
        return len(self.data)

    def _load_image(self, item):
        try:
            img_data = item.get("image")
            if img_data is None:
                raise ValueError("No image")

            # HuggingFace trả về dict {"bytes": ..., "path": ...}
            if isinstance(img_data, dict):
                img_bytes = img_data.get("bytes")
                img_path  = img_data.get("path")
                if img_bytes:
                    from io import BytesIO
                    img = Image.open(BytesIO(img_bytes)).convert("RGB")
                elif img_path:
                    img = Image.open(img_path).convert("RGB")
                else:
                    raise ValueError("Empty image dict")
            elif isinstance(img_data, (bytes, bytearray)):
                from io import BytesIO
                img = Image.open(BytesIO(img_data)).convert("RGB")
            elif isinstance(img_data, Image.Image):
                img = img_data.convert("RGB")
            else:
                img = Image.fromarray(np.array(img_data)).convert("RGB")

            return self.transform(img)

        except Exception:
            # Ảnh corrupt → trả về tensor 0
            return torch.zeros(3, 224, 224)

    def __getitem__(self, idx):
        item = self.data[idx]

        # TEXT
        text = get_text(item)
        enc  = self.tokenizer(
            str(text),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # IMAGE
        image_tensor = self._load_image(item)

        # LABEL
        label = get_label(item)

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            image_tensor,
            label
        )

In [11]:
# ============================================================
# CELL 5: Model SpotFake+
# ============================================================
class SpotFakePlus(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Text branch — XLNet
        self.xlnet   = XLNetModel.from_pretrained("xlnet-base-cased")
        self.text_fc = nn.Sequential(
            nn.Linear(768, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Image branch — VGG19
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(vgg.features.children()))
        self.image_pool     = nn.AdaptiveAvgPool2d((1, 1))
        self.image_fc       = nn.Sequential(
            nn.Linear(512, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Fusion
        self.classifier = nn.Sequential(
            nn.Linear(64, 35),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(35, num_classes)
        )

    def forward(self, input_ids, attention_mask, image_tensor):
        # Text
        text_feat = self.xlnet(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, -1, :]
        text_feat = self.text_fc(text_feat)

        # Image
        img_feat = self.image_pool(
            self.image_features(image_tensor)
        ).squeeze(-1).squeeze(-1)
        img_feat = self.image_fc(img_feat)

        # Fusion
        return self.classifier(torch.cat([text_feat, img_feat], dim=1))

In [12]:
# ============================================================
# CELL 6 (FIX): Dataloader — num_workers=0 tránh lỗi worker
# ============================================================
import datasets  # đảm bảo import

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")

train_ds = IFNDDataset(raw["train"],      tokenizer, is_train=True)
val_ds   = IFNDDataset(raw["validation"], tokenizer, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0,  # ← đổi thành 0
                          pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0,  # ← đổi thành 0
                          pin_memory=True)

print(f"Train : {len(train_ds):,} mẫu | {len(train_loader):,} steps/epoch")
print(f"Val   : {len(val_ds):,} mẫu   | {len(val_loader):,} steps/epoch")
for s in test_splits:
    print(f"Test [{s}]: {len(raw[s]):,} mẫu")

Train : 8,416 mẫu | 1,052 steps/epoch
Val   : 1,052 mẫu   | 132 steps/epoch
Test [test]: 1,053 mẫu


In [13]:
# ============================================================
# CELL 7: Training + lưu latest checkpoint
# ============================================================
model     = SpotFakePlus().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

LATEST_CKPT = os.path.join(SAVE_DIR, "latest_epoch.pth")

# Resume nếu có checkpoint
start_epoch = 1
history     = []
if os.path.exists(LATEST_CKPT):
    print("Tìm thấy checkpoint, đang resume...")
    ckpt        = torch.load(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    history     = ckpt.get("history", [])
    print(f"Tiếp tục từ epoch {start_epoch} | "
          f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}")

# Hàm evaluate
def evaluate(loader, split_name="Val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for ids, mask, imgs, labels in tqdm(loader,
                                            desc=f"  Eval [{split_name}]",
                                            leave=False):
            logits     = model(ids.to(DEVICE), mask.to(DEVICE), imgs.to(DEVICE))
            loss       = criterion(logits, labels.to(DEVICE))
            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, 1).cpu().tolist())
            all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    print(f"  [{split_name}] Loss: {total_loss/len(loader):.4f} "
          f"| Acc: {acc:.4f} | F1: {f1:.4f}")
    return total_loss / len(loader), acc, f1, all_preds, all_labels

# Training loop
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss              = 0
    train_preds, train_labels = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for ids, mask, imgs, labels in loop:
        optimizer.zero_grad()
        logits = model(ids.to(DEVICE), mask.to(DEVICE), imgs.to(DEVICE))
        loss   = criterion(logits, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(torch.argmax(logits, 1).cpu().tolist())
        train_labels.extend(labels.tolist())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc  = accuracy_score(train_labels, train_preds)
    train_f1   = f1_score(train_labels, train_preds, average="macro")
    print(f"\nEpoch {epoch} Train → "
          f"Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

    val_loss, val_acc, val_f1, _, _ = evaluate(val_loader, "Val")
    scheduler.step()

    history.append({
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "train_acc":  round(train_acc,  4),
        "train_f1":   round(train_f1,   4),
        "val_loss":   round(val_loss,   4),
        "val_acc":    round(val_acc,    4),
        "val_f1":     round(val_f1,     4),
    })

    # Lưu latest checkpoint (ghi đè sau mỗi epoch)
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "val_acc":         val_acc,
        "val_f1":          val_f1,
        "history":         history,
    }, LATEST_CKPT)
    print(f"  ✅ latest_epoch.pth đã lưu (epoch {epoch})")

print("\n=== TRAINING HOÀN TẤT ===")

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5 [Train]: 100%|██████████| 1052/1052 [12:54<00:00,  1.36it/s, loss=0.0083]



Epoch 1 Train → Loss: 0.1304 | Acc: 0.9685 | F1: 0.9550


  [Val] Loss: 0.0432 | Acc: 0.9933 | F1: 0.9907
  ✅ latest_epoch.pth đã lưu (epoch 1)


Epoch 2/5 [Train]: 100%|██████████| 1052/1052 [12:58<00:00,  1.35it/s, loss=0.0007]


...


Epoch 4 Train → Loss: 0.0265 | Acc: 0.9973 | F1: 0.9962


  [Val] Loss: 0.0407 | Acc: 0.9962 | F1: 0.9947
  ✅ latest_epoch.pth đã lưu (epoch 4)


Epoch 5/5 [Train]: 100%|██████████| 1052/1052 [13:07<00:00,  1.34it/s, loss=0.0001]



Epoch 5 Train → Loss: 0.0172 | Acc: 0.9982 | F1: 0.9975


  [Val] Loss: 0.0504 | Acc: 0.9952 | F1: 0.9933
  ✅ latest_epoch.pth đã lưu (epoch 5)

=== TRAINING HOÀN TẤT ===


In [14]:
# ============================================================
# CELL 8: Đánh giá test splits
# ============================================================
ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Load epoch {ckpt['epoch']} | "
      f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}\n")

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results = {}

for split in test_splits:
    print(f"{'='*55}")
    print(f"Split: {split}  ({len(raw[split]):,} mẫu)")
    print('='*55)

    dl = DataLoader(
        IFNDDataset(raw[split], tokenizer, is_train=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    preds_all, labels_all = [], []
    with torch.no_grad():
        for ids, mask, imgs, labels in tqdm(dl, desc=split):
            logits = model(ids.to(DEVICE), mask.to(DEVICE), imgs.to(DEVICE))
            preds_all.extend(torch.argmax(logits, 1).cpu().tolist())
            labels_all.extend(labels.tolist())

    acc    = accuracy_score(labels_all, preds_all)
    f1     = f1_score(labels_all, preds_all, average="macro")
    f1_w   = f1_score(labels_all, preds_all, average="weighted")
    report = classification_report(labels_all, preds_all,
                                   target_names=["Fake", "Real"])
    print(report)
    print(f"Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | F1 Weighted: {f1_w:.4f}\n")

    all_results[split] = {
        "accuracy":    float(acc),
        "f1_macro":    float(f1),
        "f1_weighted": float(f1_w),
        "report":      classification_report(
                           labels_all, preds_all,
                           target_names=["Fake", "Real"],
                           output_dict=True)
    }

# Bảng tóm tắt
print(f"\n{'='*67}")
print("TỔNG KẾT TEST SPLITS")
print(f"{'='*67}")
print(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}")
print("-"*67)
for split, res in all_results.items():
    print(f"{split:<35} | {res['accuracy']:>8.4f} | "
          f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}")

✅ Load epoch 5 | Val Acc: 0.9952 | Val F1: 0.9933

Split: test  (1,053 mẫu)


test: 100%|██████████| 132/132 [00:33<00:00,  3.98it/s]

              precision    recall  f1-score   support

        Fake       1.00      0.98      0.99       246
        Real       0.99      1.00      1.00       807

    accuracy                           0.99      1053
   macro avg       0.99      0.99      0.99      1053
weighted avg       0.99      0.99      0.99      1053

Accuracy: 0.9943 | F1 Macro: 0.9920 | F1 Weighted: 0.9943


TỔNG KẾT TEST SPLITS
Split                               | Accuracy | F1 Macro | F1 Weighted
-------------------------------------------------------------------
test                                |   0.9943 |   0.9920 |      0.9943


In [15]:
# ============================================================
# CELL 9: Lưu tất cả kết quả
# ============================================================
# 1. Weights model
weights_path = os.path.join(SAVE_DIR, "spotfakeplus_weights_final.pth")
torch.save(model.state_dict(), weights_path)
print(f"✅ Weights         : {weights_path}")

# 2. Full checkpoint kèm timestamp
full_ckpt_path = os.path.join(SAVE_DIR, f"spotfakeplus_full_{timestamp}.pth")
torch.save({
    "epoch":           ckpt["epoch"],
    "model_state":     model.state_dict(),
    "optimizer_state": ckpt["optimizer_state"],
    "scheduler_state": ckpt["scheduler_state"],
    "val_acc":         ckpt["val_acc"],
    "val_f1":          ckpt["val_f1"],
    "history":         ckpt["history"],
}, full_ckpt_path)
print(f"✅ Full checkpoint : {full_ckpt_path}")

# 3. JSON đầy đủ
json_data = {
    "timestamp":        timestamp,
    "checkpoint_epoch": int(ckpt["epoch"]),
    "val_acc":          float(ckpt["val_acc"]),
    "val_f1":           float(ckpt["val_f1"]),
    "training_history": ckpt["history"],
    "test_results":     all_results,
    "config": {
        "model":         "SpotFakePlus",
        "text_encoder":  "xlnet-base-cased",
        "image_encoder": "vgg19",
        "dataset":       "Nhat243/IFND-multimodal",
        "epochs":        EPOCHS,
        "batch_size":    BATCH_SIZE,
        "lr":            LR,
        "max_len":       MAX_LEN,
    }
}
json_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)
print(f"✅ JSON report     : {json_path}")

# 4. TXT dễ đọc
txt_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"SpotFake+ | IFND-multimodal | {timestamp}\n")
    f.write("=" * 67 + "\n")
    f.write(f"Model        : xlnet-base-cased + vgg19\n")
    f.write(f"Dataset      : Nhat243/IFND-multimodal\n")
    f.write(f"Epochs       : {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}\n")
    f.write(f"Checkpoint   : epoch {ckpt['epoch']}\n")
    f.write(f"Val Acc      : {ckpt['val_acc']:.4f}\n")
    f.write(f"Val F1 Macro : {ckpt['val_f1']:.4f}\n")
    f.write("\n--- Training History ---\n")
    f.write(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
            f"{'Train F1':>8} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val F1':>7}\n")
    f.write("-" * 75 + "\n")
    for h in ckpt["history"]:
        f.write(f"{h['epoch']:>6} | {h['train_loss']:>10.4f} | {h['train_acc']:>9.4f} | "
                f"{h['train_f1']:>8.4f} | {h['val_loss']:>8.4f} | "
                f"{h['val_acc']:>7.4f} | {h['val_f1']:>7.4f}\n")
    f.write(f"\n--- Test Results ---\n")
    f.write(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}\n")
    f.write("-" * 67 + "\n")
    for split, res in all_results.items():
        f.write(f"{split:<35} | {res['accuracy']:>8.4f} | "
                f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}\n")
print(f"✅ TXT report      : {txt_path}")

# 5. Liệt kê tất cả file đã lưu
print(f"\n{'='*65}")
print(f"TẤT CẢ FILE ĐÃ LƯU tại: {SAVE_DIR}")
print(f"{'='*65}")
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {fname:<50} {size:>7.1f} MB")

✅ Weights         : /content/drive/MyDrive/SpotFakePlus_IFND/spotfakeplus_weights_final.pth
✅ Full checkpoint : /content/drive/MyDrive/SpotFakePlus_IFND/spotfakeplus_full_20260408_125232.pth
✅ JSON report     : /content/drive/MyDrive/SpotFakePlus_IFND/results_all_20260408_125232.json
✅ TXT report      : /content/drive/MyDrive/SpotFakePlus_IFND/results_all_20260408_125232.txt

TẤT CẢ FILE ĐÃ LƯU tại: /content/drive/MyDrive/SpotFakePlus_IFND
  latest_epoch.pth                                    1641.5 MB
  results_all_20260408_125232.json                       0.0 MB
  results_all_20260408_125232.txt                        0.0 MB
  spotfakeplus_full_20260408_125232.pth               1641.5 MB
  spotfakeplus_weights_final.pth                       547.2 MB
